# 03 — Fine-Tune YOLO26 on BDD100K

**Goal:** Fine-tune a pretrained YOLO26 detector on BDD100K's 10 driving classes.

## Colab Runtime Notes
- **GPU**: Requires GPU runtime (T4 recommended minimum)
- **Time**: ~1-3 hours for 10 epochs on full BDD100K with T4
- **RAM**: ~12 GB GPU memory for yolo26s at imgsz=640
- **Tip**: Use `DEBUG_LIMIT` in notebook 02 for a quick test run first

## 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics>=8.4.0 opencv-python matplotlib

## 2 — GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    if gpu_mem < 8:
        print("⚠️ Low GPU memory. Consider using yolo26n instead of yolo26s.")
else:
    print("❌ No GPU! Training will be extremely slow.")
    print("→ Runtime → Change runtime type → GPU")

## 3 — Mount Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Path configuration ─────────────────────────────────────────────
ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
YOLO_DATASET_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo")
DATASET_YAML = os.path.join(YOLO_DATASET_DIR, "bdd100k_yolo.yaml")

# Where to save training outputs
TRAIN_OUTPUT_DIR = os.path.join(ECOCAR_ROOT, "training_runs")
os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)

print(f"Dataset YAML:  {DATASET_YAML}")
print(f"Output dir:    {TRAIN_OUTPUT_DIR}")

# Verify YAML exists
if os.path.isfile(DATASET_YAML):
    print("\n✅ Dataset YAML found")
    with open(DATASET_YAML, 'r') as f:
        print(f.read())
else:
    print("\n❌ Dataset YAML not found! Run notebook 02 first.")

## 4 — Training Configuration

In [ ]:
# ── Training hyperparameters ───────────────────────────────────────
# These are practical defaults for a first training run on Colab

CONFIG = {
    # Model
    "model": "yolo26s.pt",          # yolo26s for good speed/accuracy balance
                                      # Use yolo26n if GPU memory is limited
    
    # Dataset
    "data": DATASET_YAML,
    
    # Training
    "epochs": 10,                     # Start with 10; increase to 30-50 for better results
    "imgsz": 640,                     # Standard YOLO input size
    "batch": 16,                      # Reduce to 8 if OOM on T4
    "patience": 5,                    # Early stopping patience
    
    # Optimiser
    "optimizer": "auto",              # Ultralytics auto-selects best optimizer
    "lr0": 0.01,                      # Initial learning rate
    
    # Output
    "project": TRAIN_OUTPUT_DIR,
    "name": "bdd100k_yolo26s",
    "exist_ok": True,                 # Overwrite if re-running
    "save": True,
    "save_period": 5,                 # Save checkpoint every 5 epochs
    
    # Performance
    "workers": 4,                     # DataLoader workers
    "amp": True,                      # Mixed precision (faster on T4)
    "cache": False,                   # Set True if dataset fits in RAM
}

print("Training configuration:")
for k, v in CONFIG.items():
    print(f"  {k:<15} = {v}")

## 5 — Load Pretrained Model

In [ ]:
from ultralytics import YOLO

# Load COCO-pretrained YOLO26 — this gives us a strong starting point
model = YOLO(CONFIG["model"])

print(f"\n✅ Loaded pretrained model: {CONFIG['model']}")
print(f"   Original classes: {len(model.names)} (COCO)")
print(f"   Will fine-tune to: 10 classes (BDD100K)")

## 6 — Train!

In [ ]:
# Start training
# The model will automatically:
# - Replace the COCO detection head with a 10-class BDD100K head
# - Keep the pretrained backbone/neck weights (transfer learning)
# - Save best.pt and last.pt to the output directory

results = model.train(
    data=CONFIG["data"],
    epochs=CONFIG["epochs"],
    imgsz=CONFIG["imgsz"],
    batch=CONFIG["batch"],
    patience=CONFIG["patience"],
    optimizer=CONFIG["optimizer"],
    lr0=CONFIG["lr0"],
    project=CONFIG["project"],
    name=CONFIG["name"],
    exist_ok=CONFIG["exist_ok"],
    save=CONFIG["save"],
    save_period=CONFIG["save_period"],
    workers=CONFIG["workers"],
    amp=CONFIG["amp"],
    cache=CONFIG["cache"],
)

## 7 — Training Results

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Training output directory
run_dir = os.path.join(CONFIG["project"], CONFIG["name"])

print(f"\n{'='*60}")
print(f" TRAINING COMPLETE")
print(f"{'='*60}")
print(f" Output dir:  {run_dir}")

# Check for saved weights
weights_dir = os.path.join(run_dir, "weights")
if os.path.isdir(weights_dir):
    for wf in os.listdir(weights_dir):
        wpath = os.path.join(weights_dir, wf)
        size_mb = os.path.getsize(wpath) / (1024**2)
        print(f" ✅ Weights: {wf} ({size_mb:.1f} MB)")

# Show results.png if it exists
results_img = os.path.join(run_dir, "results.png")
if os.path.isfile(results_img):
    img = Image.open(results_img)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results')
    plt.tight_layout()
    plt.show()

In [ ]:
# Show confusion matrix if available
cm_img = os.path.join(run_dir, "confusion_matrix.png")
if os.path.isfile(cm_img):
    img = Image.open(cm_img)
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()

## 8 — Validate

In [ ]:
# Load best weights and run validation
best_weights = os.path.join(weights_dir, "best.pt")

if os.path.isfile(best_weights):
    best_model = YOLO(best_weights)
    val_results = best_model.val(data=DATASET_YAML)
    
    print(f"\n{'='*50}")
    print(f" Validation Results (best.pt)")
    print(f"{'='*50}")
    print(f" mAP50:    {val_results.box.map50:.4f}")
    print(f" mAP50-95: {val_results.box.map:.4f}")
    print(f"{'='*50}")
else:
    print(f"❌ best.pt not found at {best_weights}")

## 9 — Copy Best Weights to Drive (backup)

In [ ]:
import shutil

# Save a copy of best weights to an easy-to-find location
weights_backup = os.path.join(ECOCAR_ROOT, "weights")
os.makedirs(weights_backup, exist_ok=True)

if os.path.isfile(best_weights):
    dst = os.path.join(weights_backup, "bdd100k_yolo26s_best.pt")
    shutil.copy2(best_weights, dst)
    print(f"✅ Best weights backed up to: {dst}")
    print(f"\n📌 Use this path in notebooks 04, 05, 06:")
    print(f"   {dst}")
else:
    print("⚠️ No best.pt to backup")

## 10 — Summary

In [ ]:
print("\n" + "="*60)
print(" TRAINING SUMMARY")
print("="*60)
print(f" Model:      {CONFIG['model']}")
print(f" Dataset:    BDD100K (10 classes)")
print(f" Epochs:     {CONFIG['epochs']}")
print(f" Image size: {CONFIG['imgsz']}")
print(f" Batch size: {CONFIG['batch']}")
print(f" Output:     {run_dir}")
print(f" Best wts:   {best_weights}")
print("="*60)
print("\n🎯 Training complete! Proceed to notebook 04 for inference.")
print("\nTips for better results:")
print("  • Increase epochs to 30-50")
print("  • Use yolo26m.pt for higher accuracy (needs more GPU memory)")
print("  • Set cache=True if dataset fits in RAM")
print("  • Use Colab Pro for longer runtimes and better GPUs")